[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO_2026/blob/main/Orsenigo_0_information_theory.ipynb)

# Lesson 0 — Information theory: a more general perspective on the brain

**Computational Neuroscience 2026/27 · Davide Orsenigo**

The definitions are those of Cover and Thomas, *Elements of Information Theory* (2nd ed., Wiley 2006), chapters 2 and 8; the measures for more than two variables follow Rosas et al. (2019) and the review by Varley (2023). 

---
# Setup

The data are one resting-state fMRI run of one participant of the Human Connectome Project (HCP; Van Essen et
al., *NeuroImage* 80:62, 2013): 1200 volumes, one every 0.72 s (about 14 minutes), averaged within the 200
cortical regions of the Schaefer atlas (Schaefer et al., *Cereb Cortex* 28:3095, 2018). The regions are
ordered by hemisphere (left, then right) and, within each hemisphere, by the seven resting-state networks
of Yeo et al. (2011). The setup cell downloads the file (1.8 MB) from the course repository and z-scores each
region. HCP data are distributed under the WU-Minn HCP Open Access Data Use Terms.

In [ ]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy, norm

# Figure style: minimal axes, Arial-like font, palette and colormaps of figkit (D. Orsenigo)
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
PALETTE = ["#3B6E8F", "#E07A5F", "#81B29A", "#E0A458", "#8D6A9F", "#C65B6B", "#5B7080", "#3D3D3D"]
FC_CMAP = LinearSegmentedColormap.from_list(   # diverging petrol-brick, white at 0
    "fk_div", ["#2C5F86", "#6FA0BE", "#BFD3DF", "#F3F3F1", "#EFC0B2", "#D77C6A", "#B23A48"])
MI_CMAP = LinearSegmentedColormap.from_list(   # sequential cream-plum, for quantities >= 0
    "fk_warm", ["#FBEEE6", "#F6C19A", "#EE8A5C", "#D8503F", "#A4255B", "#5E1246"])
STATE_CMAP = ListedColormap(["#2C5F86", "#BFD3DF", "#EFC0B2", "#B23A48"])   # the four states 0..3
plt.rcParams.update({
    "figure.dpi": 110, "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10, "axes.titlesize": 10.5, "axes.labelsize": 10, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "mathtext.fontset": "dejavusans",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8, "axes.grid": False,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.prop_cycle": plt.cycler(color=PALETTE),
    "xtick.direction": "out", "ytick.direction": "out", "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8, "lines.linewidth": 1.4, "lines.markersize": 4.5,
    "legend.frameon": False, "image.cmap": "magma"})

# Download the data (1.8 MB) into the working directory, if they are not there yet
DATA_URL = "https://raw.githubusercontent.com/alanturin-g/Computational_Neuroscience_UNITO_2026/main/Datasets/HCP_BOLD.npz"
if not os.path.exists("HCP_BOLD.npz"):
    urllib.request.urlretrieve(DATA_URL, "HCP_BOLD.npz")
bold = np.load("HCP_BOLD.npz")["signal"]                       # shape (200 regions, 1200 volumes)
bold = (bold - bold.mean(axis=1, keepdims=True)) / bold.std(axis=1, keepdims=True)   # z-score each region
N, T = bold.shape
TR = 0.72                                                     # seconds between two volumes

# The seven networks of Yeo et al. (2011) in the order of the Schaefer atlas: regions per network,
# left hemisphere (regions 0-99), then right hemisphere (regions 100-199)
NETWORKS = ["Visual", "Somatomotor", "Dorsal attention", "Ventral attention", "Limbic", "Control", "Default"]
network = np.concatenate([np.repeat(np.arange(7), [14, 16, 13, 12, 6, 13, 26]),
                          np.repeat(np.arange(7), [15, 19, 13, 11, 6, 17, 19])])


def region_name(i):
    return f"{i} ({'RH' if i >= 100 else 'LH'} {NETWORKS[network[i]]})"


print(f"{N} regions, {T} volumes ({T * TR / 60:.1f} minutes)")

---
# Part 1 — From BOLD signals to symbols

Information theory works with probability distributions. For a variable that takes a few discrete values
(*states*) we can estimate the probability of each state by counting how often it occurs: this is the
**plug-in** estimate. The BOLD signal is continuous, so we first turn it into symbols. The simplest way is to
cut its range into bins: we cut each z-scored signal at −1, 0 and +1 standard deviations and get four states,
0 to 3. For a Gaussian signal their probabilities would be 0.16, 0.34, 0.34 and 0.16.

Discretising throws information away: two values in the same bin become the same symbol. Part 4 measures how
much.

**Demonstration.** The next cell cuts the signals into states and shows five regions.

In [ ]:
X = np.digitize(bold, [-1, 0, 1])                    # states 0, 1, 2, 3, same shape as bold
n_show = 300                                         # the first 300 volumes (3.6 minutes)
t_min = np.arange(n_show) * TR / 60

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1.25, 1]})
for i in range(5):
    axes[0].plot(t_min, bold[i, :n_show] - 5 * i, lw=0.8)
axes[0].set_yticks([-5 * i for i in range(5)], [f"region {i}" for i in range(5)])
axes[0].set_xlabel("time (min)")
axes[0].set_title("BOLD, z-scored")
im = axes[1].imshow(X[:5, :n_show], aspect="auto", cmap=STATE_CMAP, interpolation="nearest",
                    extent=[0, t_min[-1], 4.5, -0.5])
axes[1].set_yticks(range(5), [f"region {i}" for i in range(5)])
axes[1].set_xlabel("time (min)")
axes[1].set_title("four states")
cb = plt.colorbar(im, ax=axes[1], ticks=[0.375, 1.125, 1.875, 2.625], shrink=0.85)
cb.ax.set_yticklabels(["0: below −1", "1: −1 to 0", "2: 0 to 1", "3: above 1"])
plt.tight_layout()
plt.show()
print("probability of the four states, all regions together:", (np.bincount(X.ravel()) / X.size).round(3))

## Exercise 1: The entropy of one region

The entropy of a discrete variable is the average surprise of its states:

$$H(X) = -\sum_x p(x)\,\log_2 p(x), \qquad p(x) = \frac{n(x)}{\sum_{x'} n(x')},$$

where $n(x)$ is the number of time points in state $x$. States that never occur do not contribute
($p \log p \to 0$ for $p \to 0$). Write the function `entropy_bits`, which takes the array of counts and
returns $H$ in bits. Then run the next cell, which counts the states of region 0.

*Expected:* $H(X_0) \approx 1.866$ bits, below the maximum of $\log_2 4 = 2$ bits for four states and close
to the 1.90 bits of a Gaussian signal cut at the same thresholds. `scipy.stats.entropy` gives the same value
with `base=2`; without it, scipy uses natural logarithms and returns nats.

In [ ]:
def entropy_bits(counts):
    """Entropy in bits of the distribution given by counts (an array of non-negative numbers)."""
    counts = np.asarray(counts, dtype=float)
    p = ...       # YOUR CODE HERE: the probabilities of the states with a count > 0
    return ...    # YOUR CODE HERE: minus the sum of p log2 p

In [ ]:
counts0 = np.bincount(X[0], minlength=4)
print("counts of the four states in region 0:", counts0)
print(f"H(X0) = {entropy_bits(counts0):.3f} bits;  scipy: {scipy_entropy(counts0, base=2):.3f} bits "
      f"({scipy_entropy(counts0):.3f} nats without base=2)")
p_gauss = np.diff(norm.cdf([-np.inf, -1, 0, 1, np.inf]))      # a Gaussian cut at -1, 0, +1
print(f"Gaussian signal cut at the same thresholds: {entropy_bits(p_gauss):.3f} bits; maximum: 2 bits")
H_regions = np.array([entropy_bits(np.bincount(x, minlength=4)) for x in X])
print(f"over the 200 regions, H goes from {H_regions.min():.3f} to {H_regions.max():.3f} bits")

---
# Part 2 — Joint and conditional entropy

For two or more variables the state at time $t$ is the tuple of their states, and the joint entropy
$H(X, Y) = -\sum_{x,y} p(x,y) \log_2 p(x,y)$ is the entropy of the tuples. The conditional entropy
$H(X|Y)$ is the uncertainty left on $X$ once $Y$ is known, averaged over the states of $Y$. Since
$p(x,y) = p(x|y)\,p(y)$, the logarithm splits into a sum and gives the **chain rule**

$$H(X, Y) = H(X|Y) + H(Y).$$

The next cell counts the distinct joint states with `np.unique(..., axis=1)`, which looks for equal columns:
each column of the stacked signals is one time point.

In [ ]:
def joint_entropy(*signals):
    """Entropy in bits of the joint state of one or more discrete signals of the same length."""
    states = np.vstack(signals)                                 # one row per signal, one column per time point
    _, counts = np.unique(states, axis=1, return_counts=True)   # how often each joint state occurs
    return entropy_bits(counts)


print(f"one region:   H(X0) = {joint_entropy(X[0]):.3f} bits")
print(f"two regions:  H(X0, X1) = {joint_entropy(X[0], X[1]):.3f} bits")

## Demonstration: conditional entropy

**Demonstration.** The next cell uses the chain rule to compute the two conditional entropies of regions 0
and 1: $H(X_0|X_1) = H(X_0, X_1) - H(X_1)$, and the same the other way round.

*Result:* $H(X_0|X_1) \approx 1.764$ bits and $H(X_1|X_0) \approx 1.774$ bits: knowing one region removes
0.102 bits of uncertainty about the other. The two numbers differ because the two regions have
different entropies.

In [ ]:
H0, H1, H01 = joint_entropy(X[0]), joint_entropy(X[1]), joint_entropy(X[0], X[1])
H0_given_1 = H01 - H1
H1_given_0 = H01 - H0

print(f"H(X0 | X1) = {H0_given_1:.3f} bits,  H(X1 | X0) = {H1_given_0:.3f} bits")
print(f"uncertainty removed: {H0 - H0_given_1:.3f} and {H1 - H1_given_0:.3f} bits")

The uncertainty removed is the same in both directions: this is the mutual information of Part 3.

**Question 1.** Before running the next cell: can the joint entropy of regions 0–3 be smaller than that of
regions 0 and 1? An earlier version of this notebook printed 2.43 bits for the four regions and 3.64 bits for
the two. It added the terms $p \log p$ with `sum({...})`, that is, over a Python *set*.

In [ ]:
H_0123 = joint_entropy(X[0], X[1], X[2], X[3])
H_123 = joint_entropy(X[1], X[2], X[3])
print(f"H(X0, X1, X2, X3) = {H_0123:.3f} bits; the four entropies add up to "
      f"{sum(joint_entropy(X[i]) for i in range(4)):.3f} bits")
print(f"residual entropy of region 0: R(X0) = H(X0 | X1, X2, X3) = {H_0123 - H_123:.3f} bits "
      f"(H(X0) = {joint_entropy(X[0]):.3f} bits)")
print(f"joint states seen: {np.unique(X[:4], axis=1).shape[1]} of 4^4 = 256, in {T} samples")

The **residual entropy** $R(X_i) = H(X_i | X^{-i})$ is the uncertainty left on one element once all the others
are known: here the other three regions remove a third of the uncertainty of region 0. It comes back in
Part 5.

Four regions already have 256 joint states, of which 1200 samples visit 135. Ten regions would have about a
million: plug-in estimates of joint entropies of many variables are not reliable, and Part 3 shows in which
direction they fail.

---
# Part 3 — Mutual information

The mutual information between $X$ and $Y$ is the uncertainty on one variable removed by knowing the other:

$$I(X;Y) = H(X) - H(X|Y) = H(X) + H(Y) - H(X,Y) = \sum_{x,y} p(x,y)\,\log_2\frac{p(x,y)}{p(x)\,p(y)}.$$

The last form is the Kullback–Leibler divergence between the joint distribution and the product of the
marginals, the distribution the two variables would have if they were independent. It is never negative,
and it is zero only for independent variables.

## Exercise 2: Mutual information

Write `mutual_information` with the second form, using `joint_entropy`.

*Expected:* regions 4 and 5 (correlation 0.80) share $I \approx 0.562$ bits; regions 0 and 1 (correlation
0.35) share 0.102 bits, the uncertainty removed in Part 2.

In [ ]:
def mutual_information(x, y):
    """Mutual information in bits between two discrete signals (plug-in estimate)."""
    return ...    # YOUR CODE HERE

In [ ]:
for i, j in [(4, 5), (0, 1)]:
    rho = np.corrcoef(bold[i], bold[j])[0, 1]
    print(f"regions {i} and {j}: correlation {rho:.2f},  I = {mutual_information(X[i], X[j]):.3f} bits")

## Demonstration: zero correlation is not independence

The Pearson correlation measures how well a straight line fits the data. The next cell builds five pairs of
signals with 20,000 samples each. The first is linear; the other four depend on each other in other ways and
have zero correlation. Each signal is cut into 8 equally populated bins (quantiles) before computing $I$.

In [ ]:
def quantile_states(x, n_states=8):
    """Cut x into n_states equally populated bins: states 0 .. n_states-1."""
    return np.digitize(x, np.quantile(x, np.linspace(0, 1, n_states + 1)[1:-1]))


rng = np.random.default_rng(1)
n = 20_000
u, e = rng.standard_normal((2, n))
angle = rng.uniform(0, 2 * np.pi, n)
pairs = {
    "linear": (u, 0.8 * u + 0.6 * e),
    "parabola": (u, u**2 + 0.3 * e),
    "circle": (np.cos(angle) + 0.05 * u, np.sin(angle) + 0.05 * e),
    "X shape": (u, rng.choice([-1, 1], n) * u + 0.2 * e),
    "variance": (u, u * e),                   # x sets the spread of y, not its mean
}
fig, axes = plt.subplots(1, 5, figsize=(12, 2.7))
for ax, (name, (a, b)) in zip(axes, pairs.items()):
    rho = np.corrcoef(a, b)[0, 1]
    info = mutual_information(quantile_states(a), quantile_states(b))
    ax.plot(a[:2000], b[:2000], ".", ms=1.5, alpha=0.5)
    ax.set_title(f"{name}\nρ = {abs(rho) if abs(rho) < 0.005 else rho:.2f},  I = {info:.2f} bits", fontsize=9.5)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

All four nonlinear pairs have $\rho \approx 0$ and between 0.4 and 1.2 bits of mutual information. In the
last one $x$ does not change the mean of $y$, only its spread: a *modulatory* dependence, which the
correlation of the two signals cannot see (the correlation of their squares can).

## Demonstration: the bias of the estimate

With finite data the plug-in estimate of $I$ is biased upwards: two independent signals almost never give
exactly $I = 0$. The bias is measured with *surrogates*, copies of the data in which the dependence has been
destroyed on purpose:

- **shuffling** region 5 in time destroys every dependence on region 4, and also the time structure of the
  signal;
- **rolling** region 5 by a random lag (a circular shift, `np.roll`) destroys the dependence but keeps the
  autocorrelation of each signal.

**Demonstration.** The next two cells compute the MI between region 4 and 200 surrogates of region 5 of each
kind, and plot them.

*Result:* the shuffled values average about 0.005 bits, close to the formula
$(K_x - 1)(K_y - 1)/(2 N \ln 2) = 0.0054$ bits for $K = 4$ states and $N = 1200$ independent samples
(Miller 1955; Panzeri and Treves 1996). The circular shifts average about 0.02 bits, with a 95th percentile
near 0.05 bits: consecutive BOLD volumes are not independent, so the data hold fewer independent samples
than 1200. Both are far below the 0.56 bits of the real pair.

In [ ]:
rng = np.random.default_rng(0)
mi_shuffled, mi_shifted = [], []
for _ in range(200):
    mi_shuffled.append(mutual_information(X[4], rng.permutation(X[5])))
    mi_shifted.append(mutual_information(X[4], np.roll(X[5], rng.integers(50, T - 50))))

In [ ]:
bias_formula = (4 - 1) * (4 - 1) / (2 * T * np.log(2))
plt.figure(figsize=(6, 3.2))
bins = np.linspace(0, 0.08, 41)
plt.hist(mi_shuffled, bins=bins, alpha=0.8, label="shuffled")
plt.hist(mi_shifted, bins=bins, alpha=0.7, label="circular shift")
plt.axvline(bias_formula, color="#3D3D3D", ls=":", lw=1, label="formula, independent samples")
plt.xlabel("I between region 4 and a surrogate of region 5 (bits)")
plt.ylabel("surrogates")
plt.legend()
plt.show()
print(f"shuffled:       mean {np.mean(mi_shuffled):.4f} bits (formula {bias_formula:.4f}); "
      f"95th percentile {np.percentile(mi_shuffled, 95):.4f}")
print(f"circular shift: mean {np.mean(mi_shifted):.4f} bits; 95th percentile {np.percentile(mi_shifted, 95):.4f}")
print(f"real pair:      {mutual_information(X[4], X[5]):.3f} bits")

---
# Part 4 — Mutual information and functional connectivity

The functional connectivity (FC) is the matrix of Pearson correlations between the BOLD signals of all pairs
of regions. **Demonstration.** The next cell computes the matrix of mutual information between the
four-state signals of all 19,900 pairs. Instead of calling `mutual_information` 19,900 times, it counts the joint states of all pairs at
once with one matrix product.

In [ ]:
def mi_matrix(X, n_states=4):
    """Plug-in mutual information (bits) between all pairs of rows of X (states 0 .. n_states-1)."""
    n, n_t = X.shape
    onehot = (X[:, None, :] == np.arange(n_states)[None, :, None]).reshape(n * n_states, n_t).astype(float)
    P = (onehot @ onehot.T).reshape(n, n_states, n, n_states).transpose(0, 2, 1, 3) / n_t   # p_ij(a, b)
    Pi, Pj = P.sum(axis=3), P.sum(axis=2)                                                  # marginals
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = np.where(P > 0, P * np.log2(P / (Pi[..., None] * Pj[..., None, :])), 0.0)
    return terms.sum(axis=(2, 3))


FC = np.corrcoef(bold)
MI = mi_matrix(X)
iu = np.triu_indices(N, 1)                          # the 19,900 pairs (upper triangle)
print(f"check: MI[4, 5] = {MI[4, 5]:.3f} bits, as in Exercise 2")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im = axes[0].imshow(FC, vmin=-1, vmax=1, cmap=FC_CMAP)
axes[0].set_title("FC (Pearson correlation)")
plt.colorbar(im, ax=axes[0], shrink=0.8)
im = axes[1].imshow(np.where(np.eye(N, dtype=bool), np.nan, MI), vmin=0, vmax=0.5, cmap=MI_CMAP)
axes[1].set_title("mutual information (bits)")
plt.colorbar(im, ax=axes[1], shrink=0.8)
for ax in axes:
    ax.set_xlabel("region")
    ax.set_xticks([0, 100, 199])
    ax.set_yticks([0, 100, 199])
plt.tight_layout()
plt.show()
within = (network[:, None] == network[None, :])[iu]
print(f"mean FC: {FC[iu].mean():.3f};  within networks {FC[iu][within].mean():.3f}, "
      f"between networks {FC[iu][~within].mean():.3f}")

The two matrices have the same blocks: the networks of the Schaefer atlas, in each hemisphere and between
homologous regions of the two hemispheres. The MI matrix does not distinguish positive from negative
correlations: anticorrelated regions are as dependent as correlated ones.

For two jointly Gaussian variables with correlation $\rho$ the mutual information has a closed form (slides,
Part II):

$$I(X;Y) = -\tfrac{1}{2}\log_2\big(1 - \rho^2\big)\ \text{bits}.$$

## Exercise 3: Mutual information of a Gaussian pair

Write `gaussian_mi`. The next cell plots the MI of every pair of regions against its correlation, with two
curves: your formula and the MI of a Gaussian pair cut at −1, 0, +1 like the data (computed by simulation).

*Expected:* for $\rho = 0.80$, 0.737 bits. The curve of the discretised Gaussian pair lies below yours:
cutting the signals into four states loses information, and at $\rho = 0.8$ keeps 0.515 of the 0.737 bits.
The points follow the discretised curve, on average 0.02 bits above it: the bias of the circular shifts of
Part 3. Weak pairs, whose information is smaller than the bias, lie above both curves.

In [ ]:
def gaussian_mi(rho):
    """Mutual information (bits) of two jointly Gaussian variables with correlation rho."""
    return ...    # YOUR CODE HERE

In [ ]:
def discretised_gaussian_mi(rho, n=200_000, seed=0):
    """MI (bits) of a Gaussian pair with correlation rho, cut at -1, 0, +1 like the data (by simulation)."""
    a, e = np.random.default_rng(seed).standard_normal((2, n))
    b = rho * a + np.sqrt(1 - rho**2) * e
    sa, sb = np.digitize(a, [-1, 0, 1]), np.digitize(b, [-1, 0, 1])
    joint = np.bincount(4 * sa + sb, minlength=16)               # counts of the 16 joint states
    return (entropy_bits(np.bincount(sa, minlength=4)) + entropy_bits(np.bincount(sb, minlength=4))
            - entropy_bits(joint))


print(f"rho = 0.80: Gaussian {gaussian_mi(0.8):.3f} bits, cut into four states "
      f"{discretised_gaussian_mi(0.8):.3f} bits")
rho_grid = np.linspace(-0.95, 0.95, 39)
mi_cut = np.array([discretised_gaussian_mi(r) for r in rho_grid])

plt.figure(figsize=(6, 4))
plt.plot(FC[iu], MI[iu], ".", ms=1.5, alpha=0.25, color=PALETTE[0], label="pairs of regions")
plt.plot(rho_grid, gaussian_mi(rho_grid), color=PALETTE[7], lw=1.4, label="Gaussian pair")
plt.plot(rho_grid, mi_cut, color=PALETTE[1], lw=1.6, ls="--", label="Gaussian pair, four states")
plt.xlabel("correlation ρ")
plt.ylabel("mutual information (bits)")
plt.ylim(0, 1.2)
plt.legend(loc="upper center")
plt.show()
residual = MI[iu] - np.interp(FC[iu], rho_grid, mi_cut)
above, weak = MI[iu] > gaussian_mi(FC[iu]), np.abs(FC[iu]) < 0.3
print(f"distance from the discretised curve: mean {residual.mean():+.3f} bits, sd {residual.std():.3f} bits")
print(f"pairs above the Gaussian curve: {above[weak].mean():.0%} of those with |rho| < 0.3, "
      f"{above[~weak].mean():.0%} of the others")

## Demonstration: a linear Gaussian copy of the data

Is the small excess above the discretised curve a sign of nonlinear dependence, or only the bias? A sharper
test compares the data with surrogates that have the same FC and are Gaussian by construction. **Multivariate
phase randomisation** (Prichard and Theiler 1994) takes the Fourier transform of every region and adds the
same random phase to all of them at each frequency. This keeps the power spectrum of every region and all the
correlations between regions, at every lag, and destroys any other structure: the result is a linear Gaussian
process. The cell below computes the MI matrix of 10 such surrogates, and of an 11th one used as a control:
with only 10 surrogates their mean and spread are uncertain too, so even a surrogate falls further than
2 sd from them more often than 5% of the time.

In [ ]:
def phase_randomise(signals, seed=0):
    """Multivariate phase randomisation: same random phase for all signals at each frequency."""
    n_t = signals.shape[1]
    F = np.fft.rfft(signals, axis=1)
    phase = np.random.default_rng(seed).uniform(0, 2 * np.pi, F.shape[1])
    phase[0] = 0.0                                  # keep the mean
    if n_t % 2 == 0:
        phase[-1] = 0.0                             # the Nyquist term must stay real
    s = np.fft.irfft(F * np.exp(1j * phase), n=n_t, axis=1)
    return (s - s.mean(axis=1, keepdims=True)) / s.std(axis=1, keepdims=True)


surrogates = np.array([mi_matrix(np.digitize(phase_randomise(bold, seed=k), [-1, 0, 1]))[iu]
                       for k in range(10)])
mi_surr, sd_surr = surrogates.mean(axis=0), surrogates.std(axis=0)
control = mi_matrix(np.digitize(phase_randomise(bold, seed=10), [-1, 0, 1]))[iu]   # an 11th surrogate
FC_surr = np.corrcoef(phase_randomise(bold, seed=0))
print(f"FC of a surrogate against the data: largest difference {np.abs(FC_surr - FC).max():.1e}")
print(f"MI, data against surrogates: r = {np.corrcoef(MI[iu], mi_surr)[0, 1]:.3f}; "
      f"mean difference {np.mean(MI[iu] - mi_surr):+.4f} bits")
print(f"pairs further than 2 sd from the surrogates: data {np.mean(np.abs(MI[iu] - mi_surr) > 2 * sd_surr):.0%}, "
      f"11th surrogate {np.mean(np.abs(control - mi_surr) > 2 * sd_surr):.0%}")
strong = np.abs(FC[iu]) > 0.6
print(f"the {strong.sum()} pairs with |rho| > 0.6: data minus surrogates "
      f"{np.mean((MI[iu] - mi_surr)[strong]):+.4f} bits")

plt.figure(figsize=(4.2, 4))
plt.plot(mi_surr, MI[iu], ".", ms=1.5, alpha=0.25)
plt.plot([0, 1.1], [0, 1.1], color=PALETTE[7], ls=":", lw=1)
plt.xlabel("MI, Gaussian surrogates (bits)")
plt.ylabel("MI, data (bits)")
plt.show()

**Question 2.** Why does the curve of the discretised Gaussian pair lie below the continuous one? How would a
pair of regions with a strong nonlinear dependence, like the parabola of Part 3, appear on the MI–FC plot?
What do the surrogates say about a model that reproduces only the FC, such as the whole-brain model of the
next lesson?

---
# Part 5 — Beyond pairs

Mutual information has two natural extensions to $n$ variables $\mathbf{X} = (X_1, \dots, X_n)$, which
coincide with $I(X_1;X_2)$ for $n = 2$:

- the **total correlation** (Watanabe 1960) compares the whole with its parts,
  $$TC(\mathbf{X}) = \sum_{i=1}^n H(X_i) - H(\mathbf{X});$$
- the **dual total correlation** (Han 1978) is the entropy that is not residual, that is, shared by at least
  two variables,
  $$DTC(\mathbf{X}) = H(\mathbf{X}) - \sum_{i=1}^n H(X_i | \mathbf{X}^{-i}), \qquad
  H(X_i | \mathbf{X}^{-i}) = H(\mathbf{X}) - H(\mathbf{X}^{-i}).$$

Their difference, the **O-information** $\Omega = TC - DTC$ (Rosas et al. 2019), is positive when the
information is copied across many variables (**redundancy**) and negative when it exists only in the joint
state of many variables (**synergy**). Two systems of three binary signals show the extremes: three copies of
one fair coin, and two independent fair coins with their exclusive or (XOR), $X_3 = X_1 \oplus X_2$.

**Demonstration.** The next cell builds the two systems and shows their first 60 samples.

In [ ]:
rng = np.random.default_rng(2)
coin1, coin2 = rng.integers(0, 2, (2, T))
copies = np.vstack([coin1, coin1, coin1])
xor = np.vstack([coin1, coin2, coin1 ^ coin2])

fig, axes = plt.subplots(1, 2, figsize=(10, 1.6))
for ax, S, title in [(axes[0], copies, "three copies"), (axes[1], xor, "XOR: $X_3 = X_1 \\oplus X_2$")]:
    ax.imshow(S[:, :60], aspect="auto", cmap=ListedColormap(["#F3F3F1", PALETTE[0]]), interpolation="nearest")
    ax.set_yticks([0, 1, 2], ["X1", "X2", "X3"])
    ax.set_xlabel("time")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Exercise 4: Total correlation and dual total correlation

Write the two functions for an array `S` with one row per variable, using `joint_entropy(*S)` for the joint
entropy. `np.delete(S, i, axis=0)` removes row $i$, which gives $\mathbf{X}^{-i}$.

*Expected:* copies, $TC = 2.00$ and $DTC = 1.00$ bits ($\Omega = +1$); XOR, $TC = 1.00$ and $DTC = 2.00$ bits
($\Omega = -1$). In the XOR system every pair has $I < 0.01$ bits: no pairwise measure, correlation or MI,
can see it. For two variables the three measures coincide.

In [ ]:
def total_correlation(S):
    """TC = sum_i H(X_i) - H(X), in bits. S has one row per variable."""
    return ...    # YOUR CODE HERE


def dual_total_correlation(S):
    """DTC = H(X) - sum_i H(X_i | X_-i), in bits, with H(X_i | X_-i) = H(X) - H(X_-i)."""
    H_all = joint_entropy(*S)
    return ...    # YOUR CODE HERE

In [ ]:
for name, S in [("copies", copies), ("XOR", xor)]:
    tc, dtc = total_correlation(S), dual_total_correlation(S)
    print(f"{name:7s} TC = {tc:.2f}  DTC = {dtc:.2f}  O = TC - DTC = {tc - dtc:+.2f} bits")
print("XOR, MI of the three pairs:",
      ", ".join(f"{mutual_information(xor[i], xor[j]):.4f}" for i, j in [(0, 1), (0, 2), (1, 2)]), "bits")
print(f"two regions: I = {mutual_information(X[4], X[5]):.3f}, TC = {total_correlation(X[4:6]):.3f}, "
      f"DTC = {dual_total_correlation(X[4:6]):.3f} bits")

## Jointly Gaussian variables: Ω from the correlation matrix

For three regions the four-state signals have 64 joint states, and the plug-in estimate of $\Omega$ is
strongly biased (Optional C). For jointly Gaussian variables, instead, every entropy is a function of the
correlation matrix $R$, and so are the three measures:

$$TC = -\tfrac{1}{2}\log_2 \det R, \qquad
  DTC = \tfrac{1}{2}\Big[\log_2 \det R + \sum_{i=1}^n \log_2 \big(R^{-1}\big)_{ii}\Big].$$

## Demonstration: Gaussian O-information

**Demonstration.** The next cell writes these formulas as `o_information_gauss` and tests it on two Gaussian
triplets: a **common drive**, where all three correlations are 0.6, and a **sum**,
$X_3 = X_1 + X_2 + \varepsilon$ with independent $X_1, X_2$ and noise of standard deviation 0.5, so that
$\rho_{12} = 0$ and $\rho_{13} = \rho_{23} = 1/\sqrt{2.25}$.

*Result:* $\Omega = +0.213$ bits for the common drive (redundancy) and $-0.737$ bits for the sum (synergy):
$X_1$ and $X_2$ together predict $X_3$ much better than either alone.

In [ ]:
def o_information_gauss(R):
    """O-information (bits) of jointly Gaussian variables with correlation matrix R."""
    R = np.asarray(R, dtype=float)
    log_det = np.log2(np.linalg.det(R))
    tc = -0.5 * log_det
    dtc = 0.5 * (log_det + np.sum(np.log2(np.diag(np.linalg.inv(R)))))
    return tc - dtc

In [ ]:
common = np.full((3, 3), 0.6)
np.fill_diagonal(common, 1.0)
r = 1 / np.sqrt(2.25)
summed = np.array([[1, 0, r], [0, 1, r], [r, r, 1]])
print(f"common drive: O = {o_information_gauss(common):+.3f} bits")
print(f"sum:          O = {o_information_gauss(summed):+.3f} bits")

## Demonstration: redundant and synergistic triplets of brain regions

The 200 regions form 1,313,400 triplets. For three variables the Gaussian O-information has a short form in
the three correlations $a$, $b$, $c$: $\det R = 1 + 2abc - a^2 - b^2 - c^2$, and
$\Omega = \tfrac12 \log_2 \det R - \tfrac12 \big[\log_2(1-a^2) + \log_2(1-b^2) + \log_2(1-c^2)\big]$,
which is the sum of the three pairwise MIs minus the total correlation. The next cell uses it for all the
triplets at once, in about a second.

In [ ]:
J, K = np.triu_indices(N, 1)
triplets, omega = [], []
for i in range(N - 2):                               # triplets (i, j, k) with i < j < k
    keep = J > i
    j, k = J[keep], K[keep]
    a, b, c = FC[i, j], FC[i, k], FC[j, k]
    log_det = np.log2(1 + 2 * a * b * c - a**2 - b**2 - c**2)
    omega.append(0.5 * log_det - 0.5 * (np.log2(1 - a**2) + np.log2(1 - b**2) + np.log2(1 - c**2)))
    triplets.append(np.column_stack([np.full(len(j), i), j, k]))
omega, triplets = np.concatenate(omega), np.concatenate(triplets)

plt.figure(figsize=(6, 3.2))
plt.hist(omega, bins=np.linspace(-0.8, 0.8, 161), color=PALETTE[0])
plt.axvline(0, color=PALETTE[7], lw=0.8)
plt.yscale("log")
plt.xlabel("Gaussian O-information of a triplet (bits)")
plt.ylabel("triplets")
plt.show()
print(f"{len(omega):,} triplets; synergistic (O < 0): {np.mean(omega < 0):.0%}; "
      f"from {omega.min():+.2f} to {omega.max():+.2f} bits")
a, b, c = (FC[triplets[:, p], triplets[:, q]] for p, q in [(0, 1), (0, 2), (1, 2)])
odd = a * b * c < 0                                  # one or three negative correlations
print(f"triplets with an odd number of negative correlations: {odd.mean():.1%}, "
      f"and {np.mean(omega[odd] < 0):.0%} of them have O < 0")
for label, t in [("most synergistic", np.argmin(omega)), ("most redundant", np.argmax(omega))]:
    idx = triplets[t]
    print(f"\n{label}: O = {omega[t]:+.3f} bits (check: {o_information_gauss(FC[np.ix_(idx, idx)]):+.3f})")
    for i in idx:
        print("   ", region_name(i))
    print("    correlations:", np.round(FC[np.ix_(idx, idx)][np.triu_indices(3, 1)], 2))

Most triplets have $|\Omega|$ below 0.05 bits, and about half have $\Omega < 0$. At that size the sign says
little. Half of the correlations of these data are negative, and every triplet with an odd number of
negative correlations has $\Omega < 0$ (the formula above shows it: try $abc < 0$). Each correlation also
carries a statistical error of about 0.1 with 14 minutes of data. The tails are informative. The most
redundant triplets are groups of strongly correlated regions of the same system, as in a common drive. The
most synergistic ones are shaped like the Gaussian sum above: two regions that do not correlate with each
other both correlate with a third.

**Question 3.** In the most synergistic triplet, two correlations are about 0.65 and the third is about zero.
Why is it synergistic? Would you notice this triplet in the FC matrix?

---
# Optional

**A. The choice of states.** Cut the signals in two (at 0) or into 8 quantiles (`quantile_states`) and compute
again $H(X_0)$, the MI of regions 4 and 5 and the bias of Part 3. How do entropy, MI and bias change with
the number of states? Is there a best number for 1200 samples?

**B. Time: transfer entropy.** The influence of the past of $Y$ on the present of $X$, beyond what the past of
$X$ already tells, is the conditional mutual information
$$T_{Y \to X} = I(X_{t+1}; Y_t \mid X_t) = H(X_{t+1}, X_t) + H(Y_t, X_t) - H(X_t) - H(X_{t+1}, Y_t, X_t)$$
(Schreiber 2000; here with a history of one step). It is not symmetric: it measures a direction. Compute it
in both directions for regions 4 and 5 with `joint_entropy` (`X[5][1:]` is $X_{t+1}$ and `X[5][:-1]` is
$X_t$), and compare with circular-shift surrogates. For jointly Gaussian variables it equals half the Granger
causality $F$ when both are in nats (Barnett et al. 2009); in bits, $T = F / (2 \ln 2)$.

**C. The bias of the plug-in O-information.** Compute
`total_correlation(X[idx]) - dual_total_correlation(X[idx])` on the four-state signals of the most redundant
and the most synergistic triplets, and of 200 random triplets. Compare with the Gaussian values. In which
direction is the plug-in estimate biased? Hint: count the joint states that enter $TC$ and $DTC$.

---
# Summary

- Entropy measures uncertainty in bits. A BOLD signal cut into four states has 1.87 of the 2 possible bits;
  joint and conditional entropies follow from counting joint states, with the chain rule
  $H(X,Y) = H(X|Y) + H(Y)$.
- Mutual information measures dependence of any shape, and is zero only for independent variables. Finite
  data bias it upwards, more for autocorrelated signals: always compare with surrogates.
- For jointly Gaussian variables $I = -\tfrac12\log_2(1-\rho^2)$: correlation and MI carry the same information.
  At zero lag and in four states, resting-state BOLD is close to this case: a linear Gaussian copy with the
  same FC has the same MI matrix.
- Beyond pairs, total correlation and dual total correlation differ, and their difference Ω separates
  redundancy from synergy. The XOR system is invisible to every pairwise measure.
- Next lesson: a whole-brain model fitted to the FC. Near its working point the model is linear and its
  fluctuations are Gaussian, so its FC fixes every zero-lag quantity computed today. The book returns to entropy, MI,
  transfer entropy and entropy production in chapter 8 (eq. 8.1–8.4).

**Reading:** Cover and Thomas (2006), ch. 2 and 8; Timme and Lapish (2018) for neuroscience applications.

**References**
- Barnett L, Barrett AB, Seth AK (2009). Granger causality and transfer entropy are equivalent for Gaussian variables. *Phys Rev Lett* 103:238701.
- Cover TM, Thomas JA (2006). *Elements of Information Theory*, 2nd ed. Wiley.
- Deco G, Kringelbach ML (2026). *Whole-brain modelling: Cartography of the Dynamics of Mind*. Oxford University Press, open access, doi:[10.1093/oso/9780198991250.001.0001](https://doi.org/10.1093/oso/9780198991250.001.0001).
- Han TS (1978). Nonnegative entropy measures of multivariate symmetric correlations. *Inf Control* 36:133–156.
- Miller GA (1955). Note on the bias of information estimates. In: Quastler H (ed.), *Information Theory in Psychology*, Free Press, pp. 95–100.
- Panzeri S, Treves A (1996). Analytical estimates of limited sampling biases in different information measures. *Network* 7:87–107.
- Prichard D, Theiler J (1994). Generating surrogate data for time series with several simultaneously measured variables. *Phys Rev Lett* 73:951–954.
- Rosas FE, Mediano PAM, Gastpar M, Jensen HJ (2019). Quantifying high-order interdependencies via multivariate extensions of the mutual information. *Phys Rev E* 100:032305.
- Schaefer A et al. (2018). Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity MRI. *Cereb Cortex* 28:3095–3114.
- Schreiber T (2000). Measuring information transfer. *Phys Rev Lett* 85:461–464.
- Shannon CE (1948). A mathematical theory of communication. *Bell Syst Tech J* 27:379–423, 623–656.
- Timme NM, Lapish C (2018). A tutorial for information theory in neuroscience. *eNeuro* 5:ENEURO.0052-18.2018.
- Van Essen DC et al. (2013). The WU-Minn Human Connectome Project: an overview. *NeuroImage* 80:62–79.
- Varley TF (2023). Information theory for complex systems scientists: what, why, and how? arXiv:2304.12482.
- Watanabe S (1960). Information theoretical analysis of multivariate correlation. *IBM J Res Dev* 4:66–82.
- Yeo BTT et al. (2011). The organization of the human cerebral cortex estimated by intrinsic functional connectivity. *J Neurophysiol* 106:1125–1165.